# N-gram language model

In [1]:
import numpy as np
import math

## Phase 1: Dataset & Tokenization


In [2]:
corpus = [
    "I love  machine  learning",
    "I love deep learning",
    "I enjoy natural language processing",
    "Machine learning is fun",
    "Deep learning is powerful"
]
sentence = "I   love   machine learning ! !"

In [ ]:
# sentence tokenization 
def tokenize_sentence(sentence):
    puncs = ".?!,"
    sentence = sentence.lower()
    for punc in puncs:
        sentence = sentence.replace(punc,'')
    tokenized = sentence.split()
    return tokenized


In [ ]:
# corpus tokenization 
def tokenize_corpus(corpus: list[str]):
    tokenized = []
    for sentence in corpus:
        tokenized.append(tokenize_sentence(sentence))
    return tokenized 
    #  Or return [tokenize_sentence(sentence) for sentence in corpus]

In [5]:
tokenized_corpus = tokenize_corpus(corpus)

## Phase 2: vocabulary 

vocabulary is the set of all unique tokens that appear in the corpus.

In our implementation, we use Python's `set` because it automatically prevents duplicates entries.  

In [6]:
def build_vocabulary (tokenized_corpus):
    vocab = set()

    for sentence in tokenized_corpus:
        for word in sentence:
            vocab.add(word)

    return vocab

## Phase 3: Unigram

In [7]:
def count_unigrams(tokenized_corpus):
    count  = {}
    for sentence in tokenized_corpus:
        for word in sentence:
            if word in count:
                count[word] +=1
            else:
                count[word] = 1
    return (count)

In [8]:
counts_unigrams = count_unigrams(tokenized_corpus)
print(counts_unigrams)

{'i': 3, 'love': 2, 'machine': 2, 'learning': 4, 'deep': 2, 'enjoy': 1, 'natural': 1, 'language': 1, 'processing': 1, 'is': 2, 'fun': 1, 'powerful': 1}


In [9]:
def compute_unigram_probabilities(counted_unigrams:dict):
    total_words = sum(counted_unigrams.values())
    
    unigram_probabilities = {}
    
    for word, value in counted_unigrams.items():
        unigram_probabilities[word] = value/ total_words
    
    return (unigram_probabilities)

"""
Complexity

for vocabulary size V
we perform: sum(counted_unigrams.values()) 
which visits every unique word once.

So total complexity:
- Time: O(V)
- Space: O(V)
"""

'\nComplexity\n\nfor vocabulary size V\nwe perform: sum(counted_unigrams.values()) \nwhich visits every unique word once.\n\nSo total complexity:\n- Time: O(V)\n- Space: O(V)\n'

## Phase 4: N-grams function 

**Sliding window**: algorithm technique  used to process consecutive elements efficiently.  

For N-grams:
- window size = n.
- -step = 1.

Sentence:
i love machine learning

n = 2

Windows:

(i, love)  
(love, machine)  
(machine, learning)  

**Formula**:

$$
\text {Number of n-grams=L−n+1}
$$

where 
- L: length of sentence.
- $
n \leq {L}
$


In [10]:
def generate_ngrams(tokenized_corpus, n : int ):
    
    if n <= 0:
        raise ValueError("n must be greater than 0") 

    n_gram = []

    for sentence in tokenized_corpus:
        L = len(sentence)
        
        for i in range (L - n + 1):        

            n_gram.append(tuple(sentence [i:n+i]))
    
    return (n_gram)

""" 
Complexity:
- Time O(N)
- Space: O(number of generated n-grams)
"""


' \nComplexity:\n- Time O(N)\n- Space: O(number of generated n-grams)\n'

In [11]:
generated_ngrams = generate_ngrams(tokenized_corpus,2)
print(generated_ngrams)

[('i', 'love'), ('love', 'machine'), ('machine', 'learning'), ('i', 'love'), ('love', 'deep'), ('deep', 'learning'), ('i', 'enjoy'), ('enjoy', 'natural'), ('natural', 'language'), ('language', 'processing'), ('machine', 'learning'), ('learning', 'is'), ('is', 'fun'), ('deep', 'learning'), ('learning', 'is'), ('is', 'powerful')]


In [12]:
def count_ngrams(generated_ngrams):
    count = {}
    for n_gram in generated_ngrams:
        if n_gram in count: # same as "if n_gram in count.keys()" dictionaries check their keys by default.
            count[n_gram] +=1
        else:
            count[n_gram] = 1 
    
    return count

""" 
Complexity:
- Time O(M) where M is the number of generated n-grams
- Space: O(U) where U is the number of unique n-grams
"""


' \nComplexity:\n- Time O(M) where M is the number of generated n-grams\n- Space: O(U) where U is the number of unique n-grams\n'

In [13]:
print(count_ngrams(generated_ngrams))

{('i', 'love'): 2, ('love', 'machine'): 1, ('machine', 'learning'): 2, ('love', 'deep'): 1, ('deep', 'learning'): 2, ('i', 'enjoy'): 1, ('enjoy', 'natural'): 1, ('natural', 'language'): 1, ('language', 'processing'): 1, ('learning', 'is'): 2, ('is', 'fun'): 1, ('is', 'powerful'): 1}


## Phase 5: Bigrams


In [14]:
# Generate bigrams
bigrams = generate_ngrams(tokenized_corpus, 2)

In [15]:
# Count bigrams
counts_bigram = count_ngrams(bigrams)

In [ ]:
def compute_bigram_probabilities(counts_bigram,counts_unigrams ):
    prob_bigram = {}
    
    for bigram,bigram_value in counts_bigram.items():
        unigram_value = counts_unigrams[bigram[0]]
        prob_bigram[bigram] = bigram_value/unigram_value
    return prob_bigram


In [17]:
print (compute_bigram_probabilities(counts_bigram,counts_unigrams ))

{('i', 'love'): 0.6666666666666666, ('love', 'machine'): 0.5, ('machine', 'learning'): 1.0, ('love', 'deep'): 0.5, ('deep', 'learning'): 1.0, ('i', 'enjoy'): 0.3333333333333333, ('enjoy', 'natural'): 1.0, ('natural', 'language'): 1.0, ('language', 'processing'): 1.0, ('learning', 'is'): 0.5, ('is', 'fun'): 0.5, ('is', 'powerful'): 0.5}


## Phase 6: Sentence Scoring

In [18]:
sentence = ["i", "love", "machine", "learning"]

tokenized_corpus = tokenize_corpus(corpus)

counts_unigrams = count_unigrams(tokenized_corpus)

bigrams = generate_ngrams(tokenized_corpus, 2)

counts_bigram = count_ngrams(bigrams)

bigram_probabilities = compute_bigram_probabilities(counts_bigram,counts_unigrams )

print(bigram_probabilities)


{('i', 'love'): 0.6666666666666666, ('love', 'machine'): 0.5, ('machine', 'learning'): 1.0, ('love', 'deep'): 0.5, ('deep', 'learning'): 1.0, ('i', 'enjoy'): 0.3333333333333333, ('enjoy', 'natural'): 1.0, ('natural', 'language'): 1.0, ('language', 'processing'): 1.0, ('learning', 'is'): 0.5, ('is', 'fun'): 0.5, ('is', 'powerful'): 0.5}


In [19]:
def compute_sentence_probability(tokenized_sentence,bigram_probabilities ):
    L = len(tokenized_sentence)
    prob = 1

    for i in range (L - 1):
        bigram = tuple(tokenized_sentence[i:2+i]) 
        if  bigram in bigram_probabilities:
            prob = prob * bigram_probabilities[bigram]
        else:
            return 0
    return prob 
# we wont use compute_sentence_probability instead we are going to use compute_sentence_log_probability

In [20]:
sentence_probability = compute_sentence_probability(sentence,bigram_probabilities)
print(sentence_probability)

0.3333333333333333


In [ ]:
def compute_sentence_log_probability(tokenized_sentence,bigram_probabilities ):
    L = len(tokenized_sentence)
    log_prob  = 0

    for i in range (L - 1):
        bigram = tuple(tokenized_sentence[i:2+i]) 
        if  bigram in bigram_probabilities:
            p = bigram_probabilities[bigram]
            log_prob  = log_prob  + np.log(p)
        else:
            return float("-inf") # we cant return 0 as log(0) = - inf so we use this: float("-inf")
    return log_prob

In [22]:
sentence2 =  ["i", "love", "powerful", "learning"]

In [23]:
sentence_probability = compute_sentence_probability(sentence2,bigram_probabilities)
print(sentence_probability)

0


In [24]:
sentence_log_probability = compute_sentence_log_probability(sentence2,bigram_probabilities)
print(sentence_log_probability)

-inf


## Phase 7: Smoothing

In [ ]:
counts_unigrams = count_unigrams(tokenized_corpus)

counts_bigram = count_ngrams(bigrams)

vocab = build_vocabulary(tokenized_corpus)

V = len(vocab) 

In [ ]:
def compute_bigram_probabilities_smoothed(counts_bigram, counts_unigrams, vocab_size, k = 1 ):
    
    prob_bigram = {}


    for bigram,bigram_value in counts_bigram.items():
        unigram_value = counts_unigrams[bigram[0]]
        prob_bigram[bigram] = (bigram_value + k) /(unigram_value + k * vocab_size)

    return (prob_bigram)


In [51]:
bigram_probabilities_smoothed = compute_bigram_probabilities_smoothed(counts_bigram,counts_unigrams,V )
bigram_probabilities_smoothed

{('i', 'love'): 0.2,
 ('love', 'machine'): 0.14285714285714285,
 ('machine', 'learning'): 0.21428571428571427,
 ('love', 'deep'): 0.14285714285714285,
 ('deep', 'learning'): 0.21428571428571427,
 ('i', 'enjoy'): 0.13333333333333333,
 ('enjoy', 'natural'): 0.15384615384615385,
 ('natural', 'language'): 0.15384615384615385,
 ('language', 'processing'): 0.15384615384615385,
 ('learning', 'is'): 0.1875,
 ('is', 'fun'): 0.14285714285714285,
 ('is', 'powerful'): 0.14285714285714285}

In [52]:
sum(bigram_probabilities_smoothed.values())

1.9823717948717947